# EP07｜SpaceX 筷子接住星艦！火箭從太空回來，怎麼對得這麼準？

像素腦底加！這是影片裡的「火箭對準模擬器」，按上方的 **執行階段 → 全部執行**（或每格按 ▶️）就能自己跑一遍。
我們只看火箭的左右位置：一開始偏 10 公尺，路上有忽大忽小的風，看看不同的修正方法，100 次能接到幾次。

⚠️ 這是簡化的示意模型，用來看懂「回授控制」的原理，不是 SpaceX 的程式，數字也不是星艦的實際規格。

📺 影片：[youtube.com/@pixelbraintw](https://www.youtube.com/@pixelbraintw)

## 1. 風、火箭、控制
`kp` 是「偏多少」要乘的數字，`kd` 是「速度」要乘的數字，`hz` 是每秒修正幾次。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def gusts(seed, amp=0.6, T=20.0, dt=0.001, hold=2.0):
    """忽大忽小的風：每 2 秒換一次，推力在 ±amp 之間（公尺/秒²）。"""
    rng = np.random.default_rng(seed)
    n, k = int(T / dt), int(hold / dt)
    return np.repeat(rng.uniform(-amp, amp, n // k + 1), k)[:n]

def fly(wind, hz=50, kp=2.0, kd=3.0, x0=10.0, umax=3.0, tau=0.3, dt=0.001):
    """回傳每一毫秒火箭偏離目標幾公尺。hz：每秒修正幾次（None 是完全不修正）。"""
    x, v, cmd, u = x0, 0.0, 0.0, 0.0
    step = None if hz is None else int(round(1 / hz / dt))
    xs = []
    for i, w in enumerate(wind):
        if step and i % step == 0:                   # 量、比、修：每隔一段時間才做一次
            cmd = np.clip(-kp * x - kd * v, -umax, umax)
        u += (cmd - u) * dt / tau                    # 引擎轉向需要時間，慢慢跟上指令
        v += (w + u) * dt                            # 風加上引擎的推力，改變速度
        x += v * dt                                  # 速度改變位置
        xs.append(x)
    return np.array(xs)

## 2. 同一陣風，四種做法的軌跡
把 `gusts(1)` 的 1 改成別的數字，就是另一陣風。

In [ ]:
w = gusts(1)
t = np.arange(len(w)) / 1000
plt.figure(figsize=(11, 5))
for name, kw in {"no control": dict(hz=None), "position only": dict(hz=50, kd=0.0), "position + speed, 1 Hz": dict(hz=1), "position + speed, 50 Hz": dict(hz=50)}.items():
    xs = fly(w, **kw)
    plt.plot(t, xs, label=f"{name}: miss {abs(xs[-1]):.2f} m")
plt.axhline(0, color="gray", ls="--"); plt.xlabel("time (s)"); plt.ylabel("offset (m)"); plt.legend(); plt.show()

## 3. 同樣的 100 陣風，各種做法接到幾次？（誤差 0.5 公尺內算接到）
這格要跑一兩分鐘。

In [ ]:
def score(amp=0.6, **kw):
    miss = np.array([abs(fly(gusts(s, amp), **kw)[-1]) for s in range(100)])
    return int((miss < 0.5).sum()), round(float(np.median(miss)), 2)

cases = {
    "不修正": dict(hz=None),
    "只看偏多少": dict(hz=50, kd=0.0),
    "看偏多少＋看速度，每秒 1 次": dict(hz=1),
    "看偏多少＋看速度，每秒 2 次": dict(hz=2),
    "看偏多少＋看速度，每秒 50 次": dict(hz=50),
}
for name, kw in cases.items():
    caught, med = score(**kw)
    print(f"{name}：100 次接到 {caught} 次，誤差中位數 {med} 公尺")

## 4. 風變大，每秒修 50 次還接得到嗎？
試試把 `kp` 調大（例如 `score(amp, hz=50, kp=6.0, kd=5.0)`），看留下的誤差會不會變小。

In [ ]:
for amp in (0.6, 1.5, 3.0, 4.5):
    caught, med = score(amp, hz=50)
    print(f"風 ±{amp}：100 次接到 {caught} 次，誤差中位數 {med} 公尺")

---
覺得有趣的話，記得到 YouTube 訂閱 **像素腦**！有問題歡迎在影片留言 🐱